In [1]:
import pandas as pd
import numpy as np
from functools import reduce
from itertools import combinations

In [2]:
parliament = pd.read_csv("parliament.csv")

In [3]:
land = pd.read_csv("landOwners.csv")

In [4]:
savings = pd.read_csv("savings.csv")

In [5]:
youth_literacy = pd.read_csv("youthLiteracy.csv")

In [6]:
youth_literacy = youth_literacy[youth_literacy['TIME_PERIOD'] > 1999]

In [7]:
refuse_sex = pd.read_csv("beliefRefuseSex.csv")

In [8]:
refuse_sex = refuse_sex[refuse_sex['TIME_PERIOD'] > 1999]

In [9]:
belief_beating = pd.read_csv("beliefBeating.csv")

In [10]:
belief_beating = belief_beating[belief_beating['TIME_PERIOD'] > 1999]

In [11]:
family_planning = pd.read_csv("familyPlanning.csv")

In [12]:
all_data = pd.concat([parliament, land, savings, youth_literacy, refuse_sex, belief_beating, family_planning])

merged = all_data.pivot_table(
    index=['REF_AREA_LABEL', 'TIME_PERIOD'],
    columns='Area',
    values='OBS_VALUE'
).reset_index()

In [14]:
merged

Area,REF_AREA_LABEL,TIME_PERIOD,Parliament,Savings,belief beating,family planning,refuse sex,youth literacy
0,Algeria,2000,3.421053,NaN,NaN,NaN,NaN,NaN
1,Algeria,2001,3.421053,NaN,NaN,NaN,NaN,NaN
2,Algeria,2002,6.169666,NaN,NaN,NaN,NaN,0.915760
3,Algeria,2003,6.169666,NaN,NaN,NaN,NaN,NaN
4,Algeria,2004,6.169666,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1178,Zimbabwe,2018,31.481481,NaN,NaN,NaN,NaN,NaN
1179,Zimbabwe,2019,31.851852,NaN,NaN,NaN,NaN,1.117952
1180,Zimbabwe,2020,31.851852,NaN,NaN,NaN,NaN,NaN
1181,Zimbabwe,2021,31.851852,3.05,NaN,NaN,NaN,NaN


In [15]:
#Calculate correlations per country
all_correlations = []

for country, group in merged.groupby('REF_AREA_LABEL'):
    area_data = group.drop(['REF_AREA_LABEL', 'TIME_PERIOD'], axis=1)
    
    # Calculate correlation matrix
    corr_matrix = area_data.corr(min_periods=3)

    # Set diagonal to NaN (removes self-correlations)
    np.fill_diagonal(corr_matrix.values, np.nan)
    
    # Convert matrix to long format for easier CSV reading
    corr_long = corr_matrix.reset_index().melt(
        id_vars='Area',
        var_name='Area_2',
        value_name='correlation'
    )
    
    # Add country
    corr_long['country'] = country

    # Compute n_years per pair
    n_years = []
    for i, row in corr_long.iterrows():
        var1 = row['Area']
        var2 = row['Area_2']
        if var1 == var2:
            n_years.append(np.nan)
        else:
            mask = area_data[[var1, var2]].notnull().all(axis=1)
            n_years.append(mask.sum())
    corr_long['n_years'] = n_years
    
    all_correlations.append(corr_long)

In [17]:
# Combine all countries
final_results = pd.concat(all_correlations, ignore_index=True)

# Remove NaN correlations (not enough data or self-correlations)
final_results = final_results.dropna(subset=['correlation'])

# Remove duplicate pairs: keep only where INDICATOR comes before INDICATOR_2 alphabetically
final_results = final_results[
    final_results['Area'] < final_results['Area_2']
]

# Change, oldest and most recent values

In [18]:
def get_first_last(df, area_label):
    # Sort and group by area
    first = df.sort_values('TIME_PERIOD').groupby('REF_AREA_LABEL').first()[['TIME_PERIOD', 'OBS_VALUE']]
    first.columns = ['Year of First Value', 'First Value']
    last = df.sort_values('TIME_PERIOD').groupby('REF_AREA_LABEL').last()[['TIME_PERIOD', 'OBS_VALUE']]
    last.columns = ['Year of Last Value', 'Last Value']
    result = first.join(last).reset_index()
    result['Area'] = area_label
    result['Change'] = result['Last Value'] - result['First Value']
    return result

In [19]:
areas = ['parliament', 'land', 'savings', 'youth literacy', 'refuse sex', 'belief beating', 'family planning']
dfs = [parliament, land, savings, youth_literacy, refuse_sex, belief_beating, family_planning]
results = [get_first_last(df, area) for df, area in zip(dfs, areas)]

In [20]:
for df, area in zip(results, areas):
#zip(results, areas) pairs each df from results with its corresponding area from areas
    df.columns = [f"{col}_{area}" if col not in ['REF_AREA_LABEL'] else col for col in df.columns]
#reduce() is merging the multiple dfs contained in results all at once
categories_chart = reduce(lambda left, right: pd.merge(left, right, on='REF_AREA_LABEL', how='outer'), results)

In [21]:
categories_chart.to_csv("categories_chart_20260309.csv", index=False)

# Exploration

In [22]:
# We have considered correlation coefficients of >0.7 and <-0.7 indicative of "strong correlations"
# following sources such as ScienceDirect and the University of Southampton
# more on DW's repository

In [23]:
# Correlations: absolute values
final_results[(final_results['correlation'] > 0.7) | (final_results['correlation'] < -0.7)].groupby(['Area', 'Area_2'])['country'].count()

Area             Area_2         
Parliament       Savings            14
                 belief beating     11
                 family planning    11
                 refuse sex          8
                 youth literacy     21
Savings          belief beating      1
                 family planning     1
                 refuse sex          1
                 youth literacy      1
belief beating   family planning    14
                 refuse sex          6
                 youth literacy      4
family planning  refuse sex          6
                 youth literacy      7
refuse sex       youth literacy      3
Name: country, dtype: int64

In [24]:
# Finding most relevant trends
# Positive correlations expected across parameters except 'belief beating'
final_results[(final_results['correlation'] > 0.7)].groupby(['Area', 'Area_2'])['country'].count()

Area             Area_2         
Parliament       Savings            10
                 belief beating      1
                 family planning     7
                 refuse sex          4
                 youth literacy     17
Savings          family planning     1
                 youth literacy      1
belief beating   family planning     3
                 refuse sex          2
family planning  refuse sex          4
                 youth literacy      5
refuse sex       youth literacy      2
Name: country, dtype: int64

In [25]:
# Negative correlation expected between belief beating and the rest
final_results[(final_results['correlation'] < -0.7)].groupby(['Area', 'Area_2'])['country'].count()

Area             Area_2         
Parliament       Savings             4
                 belief beating     10
                 family planning     4
                 refuse sex          4
                 youth literacy      4
Savings          belief beating      1
                 refuse sex          1
belief beating   family planning    11
                 refuse sex          4
                 youth literacy      4
family planning  refuse sex          2
                 youth literacy      2
refuse sex       youth literacy      1
Name: country, dtype: int64

In [26]:
# Rank for correlation strenght by absolute values
final_results.sort_values(by='correlation', key=lambda x: x.abs(), ascending=False).head(25)

,Area,Area_2,correlation,country,n_years
1284,Parliament,refuse sex,-0.999867,Mozambique,3.0
104,belief beating,youth literacy,-0.999769,Benin,3.0
810,Parliament,family planning,0.999302,Ghana,3.0
918,Parliament,family planning,0.998210,Kenya,3.0
992,belief beating,family planning,-0.997552,Liberia,3.0
1914,Parliament,Savings,-0.996602,Zimbabwe,4.0
704,belief beating,family planning,-0.996456,Ethiopia,4.0
1495,Savings,family planning,0.996061,Senegal,3.0
948,Parliament,belief beating,-0.995958,Lesotho,3.0
717,family planning,youth literacy,0.995758,Ethiopia,4.0


In [27]:
lit_parl_corr = final_results[(final_results['correlation'] > 0.7) & (final_results['Area'] == 'Parliament') & (final_results['Area_2'] == 'youth literacy')]
lit_parl_corr

,Area,Area_2,correlation,country,n_years
30,Parliament,youth literacy,0.753969,Algeria,5.0
66,Parliament,youth literacy,0.727851,Angola,3.0
282,Parliament,youth literacy,0.842234,Cameroon,7.0
426,Parliament,youth literacy,0.711675,"Congo, Dem. Rep.",5.0
570,Parliament,youth literacy,0.813054,"Egypt, Arab Rep.",6.0
714,Parliament,youth literacy,0.719760,Ethiopia,7.0
786,Parliament,youth literacy,0.819510,"Gambia, The",4.0
858,Parliament,youth literacy,0.759066,Guinea,4.0
894,Parliament,youth literacy,0.795931,Guinea-Bissau,3.0
930,Parliament,youth literacy,0.822638,Kenya,4.0


In [28]:
# 17 paises strong positive correlation Parliament + youth literacy
# 12 SS: Angola, Cameroon, DR Congo, Ethiopia, Gambia, Guinea, Guinea-Bissau, Kenya, Mozambique, Tanzania, Uganda, Zimbabwe

In [29]:
lit_parl_corr_countries = list(lit_parl_corr['country'])

In [30]:
lit_parl_corr_countries

['Algeria',
 'Angola',
 'Cameroon',
 'Congo, Dem. Rep.',
 'Egypt, Arab Rep.',
 'Ethiopia',
 'Gambia, The',
 'Guinea',
 'Guinea-Bissau',
 'Kenya',
 'Morocco',
 'Mozambique',
 'Senegal',
 'Tanzania',
 'Tunisia',
 'Uganda',
 'Zimbabwe']

## Have countries with a positive correlation parl-literacy experienced greater positive change than the others?
It looks like it!

In [31]:
categories_chart[categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['Change_youth literacy'].mean()

np.float64(0.17975983549566843)

In [32]:
categories_chart[~categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['Change_youth literacy'].mean()

np.float64(0.10401784011295917)

In [33]:
corr_first_value_yl = categories_chart[categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['First Value_youth literacy'].mean()
corr_last_value_yl = categories_chart[categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['Last Value_youth literacy'].mean()
corr_first_value_parl = categories_chart[categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['First Value_parliament'].mean()
corr_last_value_parl = categories_chart[categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['Last Value_parliament'].mean()

In [34]:
no_corr_first_value_yl = categories_chart[~categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['First Value_youth literacy'].mean()
no_corr_last_value_yl = categories_chart[~categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['Last Value_youth literacy'].mean()
no_corr_first_value_parl = categories_chart[~categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['First Value_parliament'].mean()
no_corr_last_value_parl = categories_chart[~categories_chart['REF_AREA_LABEL'].isin(lit_parl_corr_countries)]['Last Value_parliament'].mean()

In [35]:
yl_parl_corr_no_corr = pd.DataFrame(
    data=[
        [corr_first_value_yl, corr_last_value_yl],           
        [no_corr_first_value_yl, no_corr_last_value_yl],     
        [corr_first_value_parl, corr_last_value_parl],       
        [no_corr_first_value_parl, no_corr_last_value_parl], 
    ],
    index=['Corr countries yl', 'No corr countries yl', 'Corr countries parl', 'No corr countries parl'],
    columns=["First value", "Last value"]
)

In [36]:
yl_parl_corr_no_corr

,First value,Last value
Corr countries yl,0.795950,0.975710
No corr countries yl,0.840121,0.944138
Corr countries parl,10.118797,27.711910
No corr countries parl,10.057496,22.459389


In [37]:
final_results[(final_results['correlation'] < -0.7) & (final_results['Area'] == 'Parliament') & (final_results['Area_2'] == 'belief beating')]
# 10 correlations between belief beating vs parliament

,Area,Area_2,correlation,country,n_years
84,Parliament,belief beating,-0.948582,Benin,4.0
264,Parliament,belief beating,-0.994024,Cameroon,3.0
696,Parliament,belief beating,-0.940150,Ethiopia,4.0
804,Parliament,belief beating,-0.828086,Ghana,4.0
912,Parliament,belief beating,-0.960270,Kenya,4.0
948,Parliament,belief beating,-0.995958,Lesotho,3.0
1092,Parliament,belief beating,-0.967109,Malawi,4.0
1272,Parliament,belief beating,-0.989975,Mozambique,3.0
1416,Parliament,belief beating,-0.809042,Rwanda,5.0
1848,Parliament,belief beating,-0.870460,Uganda,4.0


In [38]:
# An exception
final_results[(final_results['correlation'] > 0.7) & (final_results['Area_2'] == 'belief beating')]

,Area,Area_2,correlation,country,n_years
1128,Parliament,belief beating,0.755113,Mali,3.0


In [39]:
# more exceptions
final_results[(final_results['correlation'] > 0.7) & (final_results['Area'] == 'belief beating')]

,Area,Area_2,correlation,country,n_years
92,belief beating,family planning,0.982577,Benin,4.0
272,belief beating,family planning,0.957353,Cameroon,3.0
854,belief beating,refuse sex,0.981034,Guinea,3.0
1064,belief beating,family planning,0.951190,Madagascar,3.0
1286,belief beating,refuse sex,0.992147,Mozambique,3.0


In [40]:
categories_chart[categories_chart['Last Value_parliament'] < 4].sort_values(by='Last Value_parliament', ascending=False)

,REF_AREA_LABEL,Year of First Value_parliament,First Value_parliament,Year of Last Value_parliament,Last Value_parliament,Area_parliament,Change_parliament,Year of First Value_land,First Value_land,Year of Last Value_land,...,Year of Last Value_belief beating,Last Value_belief beating,Area_belief beating,Change_belief beating,Year of First Value_family planning,First Value_family planning,Year of Last Value_family planning,Last Value_family planning,Area_family planning,Change_family planning
38,Nigeria,2000,3.418803,2022,3.611111,parliament,0.192308,2013.0,4.788889,2018.0,...,2018.0,28.0,belief beating,-36.2,2003.0,41.8,2018.0,46.9,family planning,5.1


In [42]:
categories_chart[['REF_AREA_LABEL', 'Year of First Value_parliament',
       'First Value_parliament', 'Year of Last Value_parliament',
       'Last Value_parliament', 'Year of First Value_youth literacy',
       'First Value_youth literacy', 'Year of Last Value_youth literacy',
       'Last Value_youth literacy']] 

,REF_AREA_LABEL,Year of First Value_parliament,First Value_parliament,Year of Last Value_parliament,Last Value_parliament,Year of First Value_youth literacy,First Value_youth literacy,Year of Last Value_youth literacy,Last Value_youth literacy
0,Algeria,2000,3.421053,2022,8.108108,2002.0,0.915760,2019.0,1.230157
1,Angola,2000,15.454545,2022,33.636364,2001.0,0.754170,2022.0,0.939460
2,Benin,2000,6.024096,2022,7.407407,2001.0,0.521447,2022.0,0.802990
3,Botswana,2000,17.021277,2022,11.111111,2003.0,1.036750,2013.0,1.034140
4,Burkina Faso,2000,8.108108,2022,16.901408,2003.0,0.646450,2022.0,0.930170
5,Burundi,2000,14.406780,2022,38.211382,2000.0,0.916450,2022.0,0.991380
6,Cabo Verde,2000,11.111111,2022,38.888889,2000.0,1.015730,2022.0,1.010080
7,Cameroon,2000,5.555556,2022,33.888889,2000.0,0.881700,2020.0,0.945840
8,Central African Republic,2000,7.339450,2022,12.857143,2000.0,0.675250,2020.0,0.612200
9,Chad,2000,2.400000,2022,25.888325,2000.0,0.417260,2022.0,0.740280
